In [1]:
import pandas as pd
import numpy as np
import sqlite3
from datetime import datetime, timedelta


np.random.seed(123)
total_customers = 200
total_orders = 1500


customer_ids = [f"CUST_{i:03d}" for i in range(1, total_customers + 1)]
signup_dates = [datetime(2022, 1, 1) + timedelta(days=int(i)) for i in np.random.randint(0, 365, total_customers)]
registration_cities = np.random.choice(["Москва", "Санкт-Петербург", "Казань", "Екатеринбург", "Новосибирск"], total_customers)

customers_df = pd.DataFrame({
    "customer_id": customer_ids,
    "signup_date": signup_dates,
    "city": registration_cities
})


order_ids = [f"ORD_{i:05d}" for i in range(1, total_orders + 1)]
order_customer_ids = np.random.choice(customer_ids, total_orders)
order_dates = [datetime(2023, 1, 1) + timedelta(days=int(i)) for i in np.random.randint(0, 700, total_orders)]
order_amounts = np.round(np.random.lognormal(mean=9.5, sigma=1.0, size=total_orders), 2)
order_statuses = np.random.choice(["completed", "completed", "completed", "canceled", "refunded"], total_orders)

orders_df = pd.DataFrame({
    "order_id": order_ids,
    "customer_id": order_customer_ids,
    "order_date": order_dates,
    "amount": order_amounts,
    "status": order_statuses
})


product_ids = [f"PROD_{i:03d}" for i in range(1, 31)]
product_names = [f"Товар_{i}" for i in range(1, 31)]
product_categories = np.random.choice(["Электроника", "Одежда", "Дом", "Спорт", "Книги"], 30)

products_df = pd.DataFrame({
    "product_id": product_ids,
    "product_name": product_names,
    "category": product_categories
})


connection = sqlite3.connect("customers.db")

customers_df.to_sql("customers", connection, if_exists="replace", index=False)
orders_df.to_sql("orders", connection, if_exists="replace", index=False)
products_df.to_sql("products", connection, if_exists="replace", index=False)

connection.commit()


rfm_query = """
WITH customer_metrics AS (
    SELECT
        c.customer_id,
        c.city,
        COUNT(o.order_id) AS total_orders,
        ROUND(SUM(o.amount), 2) AS total_spent,
        CAST(julianday('2024-12-01') - julianday(MAX(o.order_date)) AS INTEGER) AS days_since_last_order
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id AND o.status = 'completed'
    GROUP BY c.customer_id, c.city
),
ranked_customers AS (
    SELECT
        customer_id,
        city,
        total_orders,
        total_spent,
        days_since_last_order,
        NTILE(4) OVER (ORDER BY days_since_last_order DESC) AS recency_rank,
        NTILE(4) OVER (ORDER BY total_orders) AS frequency_rank,
        NTILE(4) OVER (ORDER BY total_spent) AS monetary_rank
    FROM customer_metrics
    WHERE total_orders > 0
)
SELECT
    customer_id,
    city,
    total_orders,
    total_spent,
    days_since_last_order,
    recency_rank,
    frequency_rank,
    monetary_rank,
    CASE
        WHEN recency_rank <= 2 AND frequency_rank >= 3 AND monetary_rank >= 3 THEN 'VIP'
        WHEN recency_rank <= 2 AND frequency_rank >= 3 THEN 'Лояльные'
        WHEN recency_rank >= 3 THEN 'Риск ухода'
        WHEN frequency_rank <= 2 AND monetary_rank <= 2 THEN 'Новички'
        ELSE 'Средний сегмент'
    END AS segment
FROM ranked_customers
ORDER BY total_spent DESC
"""

rfm_results = pd.read_sql_query(rfm_query, connection)

print("--- RFM-СЕГМЕНТАЦИЯ КЛИЕНТОВ ---")
display(rfm_results.head(10))

print("\n--- РАСПРЕДЕЛЕНИЕ ПО СЕГМЕНТАМ ---")
segment_distribution = rfm_results["segment"].value_counts()
display(segment_distribution)

print("\n--- СРЕДНИЙ ЧЕК ПО СЕГМЕНТАМ ---")
segment_metrics = rfm_results.groupby("segment").agg(
    avg_orders=("total_orders", "mean"),
    avg_spent=("total_spent", "mean"),
    avg_recency=("days_since_last_order", "mean")
).round(2)
display(segment_metrics)

rfm_results.to_csv("rfm_segmentation.csv", index=False)
print("\nРезультаты сохранены в rfm_segmentation.csv")

connection.close()

--- RFM-СЕГМЕНТАЦИЯ КЛИЕНТОВ ---


,customer_id,city,total_orders,total_spent,days_since_last_order,recency_rank,frequency_rank,monetary_rank,segment
0,CUST_017,Санкт-Петербург,8,390815.28,7,4,4,4,Риск ухода
1,CUST_177,Казань,6,335474.16,125,2,4,4,VIP
2,CUST_089,Новосибирск,6,299177.61,93,3,4,4,Риск ухода
3,CUST_048,Москва,4,289386.50,12,4,3,4,Риск ухода
4,CUST_159,Санкт-Петербург,10,269461.41,69,3,4,4,Риск ухода
5,CUST_056,Санкт-Петербург,8,264509.13,41,4,4,4,Риск ухода
6,CUST_076,Москва,6,242393.76,22,4,4,4,Риск ухода
7,CUST_154,Новосибирск,9,238386.77,38,4,4,4,Риск ухода
8,CUST_024,Санкт-Петербург,5,232628.03,178,2,3,4,VIP
9,CUST_182,Казань,5,223980.30,343,1,3,4,VIP



--- РАСПРЕДЕЛЕНИЕ ПО СЕГМЕНТАМ ---


,count
segment,
Риск ухода,98
Новички,48
VIP,38
Лояльные,7
Средний сегмент,6



--- СРЕДНИЙ ЧЕК ПО СЕГМЕНТАМ ---


,avg_orders,avg_spent,avg_recency
segment,,,
VIP,5.53,159557.36,214.55
Лояльные,5.29,66172.26,222.57
Новички,2.71,40064.17,309.56
Риск ухода,4.89,117104.82,52.54
Средний сегмент,3.00,136856.12,312.00



Результаты сохранены в rfm_segmentation.csv
